# 🔘🖼️ Gradio FLUX Depth

⚠️ **Remember to copy this notebook in your Drive and rename.**

🤗 This notebook uses [Hugging Face Diffusers](https://huggingface.co/docs/diffusers/en/index) to create pipelines for tasks such as image generation.

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

Gradio's [Documentation](https://www.gradio.app/docs)

## Install Gradio (and Restart)

In [1]:
!pip install gradio --quiet

### Mount Drive

In [2]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Hugging Face Token

In [3]:
# Sign up at Hugging Face and create a "Read" access token (not the default "Fine-Grained" token).
# Click the 🔑 "Secrets" icon in the left sidebar.
# Enable Notebook Access, Set the Name to "HF_TOKEN", Paste your token as the Value

import getpass
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token

print("HF_TOKEN environment variable has been set.")

HF_TOKEN environment variable has been set.


## Setup

In [4]:
%cd /content
!rm -rf iaac_genai
!git clone https://github.com/jamesmcbennett/iaac_genai
%cd /content/iaac_genai/

/content
Cloning into 'iaac_genai'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 17 (delta 6), reused 12 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 5.37 KiB | 5.37 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/iaac_genai


In [5]:
import sys
sys.path.append('/content/iaac_genai')

In [6]:
!pip install -q -r requirements.txt --quiet > /dev/null 2>&1

In [7]:
from config import Config
from utils import set_image_path, save_image, save_yml, save_svg
import torch

from diffusers.utils import load_image
from PIL import Image, ImageDraw, ImageFont
from transformers import pipeline as hf_pipeline
import gradio as gr
import numpy as np
import random
import os
import cv2

In [8]:
# !pip install -q --upgrade diffusers transformers accelerate huggingface_hub peft torchao
!pip install -q --upgrade accelerate huggingface_hub peft torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 112.6 MB/s eta 0:00:00


## Output Directory

In [9]:
Config.OUTPUT_DIR = "/content/drive/MyDrive/output"
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

## Set LoRA Path

In [26]:
LORA_REPO = 'strangerzonehf/Ghibli-Flux-Cartoon-LoRA'
LORA_WEIGHT = 'Ghibili-Cartoon-Art.safetensors'
TRIGGER = 'Ghibli Art –'

## Load Flux ControlNet Pipeline

In [27]:
from diffusers import FluxControlNetPipeline, FluxControlNetModel, FluxMultiControlNetModel
from safetensors.torch import load_file

base_model = "black-forest-labs/FLUX.1-dev"
controlnet_model = "Shakker-Labs/FLUX.1-dev-ControlNet-Depth"

controlnet = FluxControlNetModel.from_pretrained(controlnet_model, torch_dtype=torch.bfloat16)
pipe = FluxControlNetPipeline.from_pretrained(base_model, controlnet=controlnet, torch_dtype=torch.bfloat16)
pipe.to("cuda")

# LoRA
pipe.unload_lora_weights()
pipe.load_lora_weights(LORA_REPO, weight_name=LORA_WEIGHT)

# Depth
depth_estimator = hf_pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=0)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/diffusers/models/embeddings.py:2616: FutureWarning: `FluxPosEmbed` is deprecated and will be removed in version 1.0.0. Importing and using `FluxPosEmbed` from `diffusers.models.embeddings` is deprecated. Please import it from `diffusers.models.transformers.transformer_flux`.
  deprecate("FluxPosEmbed", "1.0.0", deprecation_message)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Ghibili-Cartoon-Art.safetensors:   0%|          | 0.00/613M [00:00<?, ?B/s]

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

## Functions

In [28]:
# Resize helper
def resize(image, max_dim=1024):
    w, h = image.size
    scale = max_dim / max(w, h)
    return image.resize((int(w * scale), int(h * scale)))

# Generate depth map
def get_depth(image: Image.Image) -> Image.Image:
    image = resize(image)
    depth_image = depth_estimator(image)['depth']
    depth_np = np.array(depth_image)
    depth_np = (depth_np - depth_np.min()) / (depth_np.max() - depth_np.min())
    depth_np = (depth_np * 255).astype(np.uint8)
    depth_rgb = np.stack([depth_np] * 3, axis=2)
    return Image.fromarray(depth_rgb)

import gc

def generate_flux_depth_only(
    image, prompt, guidance, steps, controlnet_scale
):
    gc.collect()
    torch.cuda.empty_cache()

    # Resize input and generate normalized depth map
    image_resized = resize(image, 1024)
    control_image = get_depth(image_resized).resize((1024, 1024)).convert("RGB")
    control_mode = 2  # Depth

    width, height = control_image.size

    result = pipe(
        prompt=prompt,
        control_image=control_image,
        control_mode=control_mode,
        width=width,
        height=height,
        controlnet_conditioning_scale=float(controlnet_scale),
        control_guidance_start=0.0,
        control_guidance_end=1.0,
        num_inference_steps=steps,
        guidance_scale=guidance,
        max_sequence_length=512,
        joint_attention_kwargs={"scale": 1.0},
    ).images[0]

    return control_image, result

## Run Gradio

In [29]:
demo = gr.Interface(
    fn=generate_flux_depth_only,
    inputs=[
        gr.Image(type="pil", label="Input Image"),
        gr.Textbox(label="Prompt", value=TRIGGER + " " + "postmodern Mediterranean apartment complex, stepped terraces and geometric staircases, red and ochre rendered walls, arched doorways, swimming pool in foreground, sun loungers, people relaxing poolside, warm afternoon sunlight, strong shadows, lush palm trees, coastal resort, wide angle, architectural photography"),
        gr.Slider(1.0, 15.0, step=0.5, value=3.5, label="Guidance Scale"),
        gr.Slider(10, 50, step=1, value=28, label="Inference Steps"),
        gr.Slider(0.1, 2.0, step=0.01, value=0.6, label="ControlNet Conditioning Scale"),
    ],
    outputs=[
        gr.Image(type="pil", label="Depth Control Image"),
        gr.Image(type="pil", label="Generated Output"),
    ],
    title="Flux.1 + LoRA + ControlNet-Union (Depth Only)",
    description="Upload an image and prompt. Uses Flux.1 with DreamBooth LoRA and depth-based conditioning via ControlNet-Union.",
)

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8f7a99080316b73320.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
